# Практическое занятие №3: Свёртка и корреляционный анализ сигналов

## 1. Функции Python для свёртки и корреляции

### 1.1. `np.convolve(x, h, mode)`

- `mode='full'` – полная линейная свёртка (длина $N_x+N_h-1$).
- `mode='same'` – центральная часть той же длины, что и $x$.
- `mode='valid'` – только те точки, где ядро полностью перекрывается с сигналом.

### 1.2. `np.correlate(x, y, mode)`

Аналогичные режимы. При `mode='full'` результат имеет длину $N_x+N_y-1$. Индекс 0 соответствует полному перекрытию сигналов.

Для получения сдвига, при котором достигается максимум корреляции, используется формула:
```python
corr = np.correlate(x, y, mode='full')
delay = np.argmax(corr) - (len(x) - 1)
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile

np.random.seed(42)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True


## 2. Гауссовы окна и их свёртка

Гауссово окно можно сгенерировать с помощью `scipy.signal.windows.gaussian(M, std)`.

**Аналитическое свойство:** свёртка двух гауссовых функций с дисперсиями $\sigma_1^2$ и $\sigma_2^2$ даёт гауссову функцию с дисперсией $\sigma_1^2+\sigma_2^2$:
$$
(f_{\sigma_1} * f_{\sigma_2})(t) = f_{\sqrt{\sigma_1^2+\sigma_2^2}}(t).
$$


In [ ]:
sigma1 = 3
sigma2 = 5

M1 = 10 * sigma1 + 1
M2 = 10 * sigma2 + 1

g1 = signal.windows.gaussian(M1, std=sigma1)
g2 = signal.windows.gaussian(M2, std=sigma2)
g1 = g1 / np.sum(g1)
g2 = g2 / np.sum(g2)

conv_num = np.convolve(g1, g2, mode="full")

n1 = np.arange(-(M1 // 2), M1 // 2 + 1)
n2 = np.arange(-(M2 // 2), M2 // 2 + 1)
n_conv = np.arange(n1[0] + n2[0], n1[-1] + n2[-1] + 1)

sigma_theory = np.sqrt(sigma1 ** 2 + sigma2 ** 2)
g_theory = np.exp(-(n_conv ** 2) / (2 * sigma_theory ** 2))
g_theory = g_theory / np.sum(g_theory)

mse = np.mean((conv_num - g_theory) ** 2)

print(f"Стандартное отклонение теоретической функции: {sigma_theory:.4f}")
print(f"Среднеквадратичная ошибка: {mse:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(n1, g1, label="Гауссово окно 1")
axes[0].plot(n2, g2, label="Гауссово окно 2")
axes[0].set_title("Исходные функции")
axes[0].set_xlabel("Отсчёт")
axes[0].set_ylabel("Амплитуда")
axes[0].legend()

axes[1].plot(n_conv, conv_num, label="Результат свёртки (численный)", linewidth=2)
axes[1].plot(n_conv, g_theory, "--", label="Теоретическая гауссова функция", linewidth=2)
axes[1].set_title("Сравнение численного и аналитического результатов")
axes[1].set_xlabel("Отсчёт")
axes[1].set_ylabel("Амплитуда")
axes[1].legend()

plt.tight_layout()
plt.show()


**вывод: численная свёртка двух нормированных гауссовых окон совпадает с теоретической гауссовой функцией с дисперсией \\(\sigma_1^2 + \sigma_2^2\\), а ошибка получается пренебрежимо малой.**


## 3. Фильтрация с помощью свёртки

**Прямоугольное окно (усреднение):**
```python
h_rect = signal.windows.boxcar(L) / L
```

**Гауссовское окно:**
```python
h_gauss = signal.windows.gaussian(L, std=L/5)
h_gauss = h_gauss / np.sum(h_gauss)
```


In [ ]:
fs = 1000
duration = 2
t = np.arange(0, duration, 1 / fs)
x = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 80 * t)

window_lengths = [11, 21, 41, 81]
time_mask = t <= 0.5

def fft_amplitudes(sig):
    freqs = np.fft.rfftfreq(len(sig), d=1 / fs)
    spectrum = 2 * np.abs(np.fft.rfft(sig)) / len(sig)
    idx_5 = np.argmin(np.abs(freqs - 5))
    idx_80 = np.argmin(np.abs(freqs - 80))
    return spectrum[idx_5], spectrum[idx_80]

print("Сравнение амплитуд после фильтрации")
print("L | 5 Гц после прямоугольного | 80 Гц после прямоугольного | 5 Гц после гауссовского | 80 Гц после гауссовского")

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
axes = axes.ravel()

for i, L in enumerate(window_lengths):
    h_rect = signal.windows.boxcar(L) / L
    h_gauss = signal.windows.gaussian(L, std=L / 5)
    h_gauss = h_gauss / np.sum(h_gauss)

    x_rect = np.convolve(x, h_rect, mode="same")
    x_gauss = np.convolve(x, h_gauss, mode="same")

    amp_rect_5, amp_rect_80 = fft_amplitudes(x_rect)
    amp_gauss_5, amp_gauss_80 = fft_amplitudes(x_gauss)

    print(
        f"{L:>2} | {amp_rect_5:>24.4f} | {amp_rect_80:>25.4f} | "
        f"{amp_gauss_5:>23.4f} | {amp_gauss_80:>24.4f}"
    )

    axes[i].plot(t[time_mask], x[time_mask], label="Исходный сигнал", linewidth=1.5)
    axes[i].plot(t[time_mask], x_rect[time_mask], label="Прямоугольное окно")
    axes[i].plot(t[time_mask], x_gauss[time_mask], label="Гауссовское окно")
    axes[i].set_title(f"Длина окна L = {L}")
    axes[i].set_xlabel("Время, с")
    axes[i].set_ylabel("Амплитуда")
    axes[i].legend()

plt.tight_layout()
plt.show()


**вывод: при увеличении длины окна высокочастотная составляющая 80 Гц подавляется сильнее, а гауссовское окно при тех же длинах лучше сохраняет полезную составляющую 5 Гц, чем прямоугольное.**


## 4. Поиск временной задержки с помощью кросс-корреляции

Для воспроизводимости используется `np.random.seed(42)`, а при анализе влияния шума результаты усредняются по нескольким значениям `seed`.


In [ ]:
fs = 1000
delay = 100
noise_levels = np.linspace(0, 2, 11)
seeds = range(42, 62)

def generate_signal(seed):
    rng = np.random.default_rng(seed)
    freqs = rng.uniform(10, 100, 100)
    amps = rng.uniform(0.5, 1.5, 100)
    phases = rng.uniform(0, 2 * np.pi, 100)
    t_local = np.arange(0, 1, 1 / fs)
    x_local = np.sum(
        amps[:, None] * np.sin(2 * np.pi * freqs[:, None] * t_local + phases[:, None]),
        axis=0,
    )
    return t_local, x_local / np.max(np.abs(x_local))

mean_abs_error = []
exact_rate = []

for noise_level in noise_levels:
    errors = []
    for seed in seeds:
        _, x_local = generate_signal(seed)
        y_local = np.zeros_like(x_local)
        y_local[delay:] = x_local[:-delay]
        rng_noise = np.random.default_rng(seed + 1000)
        y_local = y_local + noise_level * rng_noise.standard_normal(len(x_local))

        corr = np.correlate(y_local, x_local, mode="full")
        delay_est = int(np.argmax(corr) - (len(x_local) - 1))
        errors.append(abs(delay_est - delay))

    errors = np.array(errors)
    mean_abs_error.append(errors.mean())
    exact_rate.append(np.mean(errors == 0))

print("Усреднённая оценка по 20 прогонам")
print("Уровень шума | Средняя абсолютная ошибка | Доля точного определения")
for noise_level, err, rate_ok in zip(noise_levels, mean_abs_error, exact_rate):
    print(f"{noise_level:>12.1f} | {err:>27.3f} | {rate_ok:>24.3f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(noise_levels, mean_abs_error, marker="o")
axes[0].set_title("Средняя абсолютная ошибка оценки задержки")
axes[0].set_ylabel("Ошибка, отсчёты")

axes[1].plot(noise_levels, exact_rate, marker="s")
axes[1].set_title("Доля точного определения задержки")
axes[1].set_xlabel("Уровень шума")
axes[1].set_ylabel("Доля точных оценок")

plt.tight_layout()
plt.show()

example_noise_levels = [0.3, 2.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, noise_level in enumerate(example_noise_levels):
    t_example, x_example = generate_signal(42)
    y_example = np.zeros_like(x_example)
    y_example[delay:] = x_example[:-delay]
    rng_noise = np.random.default_rng(1042 + col)
    y_example = y_example + noise_level * rng_noise.standard_normal(len(y_example))

    corr_example = np.correlate(y_example, x_example, mode="full")
    lags = np.arange(-(len(x_example) - 1), len(y_example))
    delay_est = int(np.argmax(corr_example) - (len(x_example) - 1))

    axes[0, col].plot(t_example[:300], x_example[:300], label="x")
    axes[0, col].plot(t_example[:300], y_example[:300], label="y")
    axes[0, col].set_title(f"Сигналы при уровне шума {noise_level}")
    axes[0, col].set_xlabel("Время, с")
    axes[0, col].set_ylabel("Амплитуда")
    axes[0, col].legend()

    axes[1, col].plot(lags, corr_example, label="Кросс-корреляция")
    axes[1, col].axvline(delay, color="g", linestyle=":", label="Истинная задержка")
    axes[1, col].axvline(delay_est, color="r", linestyle="--", label="Найденная задержка")
    axes[1, col].set_title(f"Кросс-корреляция при уровне шума {noise_level}")
    axes[1, col].set_xlabel("Задержка, отсчёты")
    axes[1, col].set_ylabel("Корреляция")
    axes[1, col].legend()

plt.tight_layout()
plt.show()


**вывод: максимум кросс-корреляции корректно восстанавливает задержку 100 отсчётов, а при росте уровня шума точность постепенно снижается, поэтому усреднение по нескольким `seed` даёт устойчивую оценку ошибки.**


## 5. Обнаружение шаблона в зашумлённом сигнале

Для анализа влияния отношения сигнал/шум результаты также усредняются по нескольким запускам с разными `seed`.


In [ ]:
fs = 1000
sigma = 100
template_len = 6 * sigma
long_len = 2000
amplitudes = np.arange(0.2, 5.01, 0.4)
seeds = range(42, 62)

n_template = np.arange(template_len) - template_len // 2
envelope = np.exp(-(n_template ** 2) / (2 * sigma ** 2))
template = envelope * np.sin(2 * np.pi * 20 * np.arange(template_len) / fs)

mean_abs_error = []
exact_rate = []

for amplitude in amplitudes:
    errors = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        long_signal = rng.standard_normal(long_len)
        true_pos = int(rng.integers(0, long_len - template_len + 1))
        long_signal[true_pos:true_pos + template_len] += amplitude * template

        corr = np.correlate(long_signal, template, mode="valid")
        est_pos = int(np.argmax(corr))
        errors.append(abs(est_pos - true_pos))

    errors = np.array(errors)
    mean_abs_error.append(errors.mean())
    exact_rate.append(np.mean(errors == 0))

print("Усреднённая оценка по 20 прогонам")
print("Амплитуда шаблона | Средняя абсолютная ошибка | Доля точного обнаружения")
for amplitude, err, rate_ok in zip(amplitudes, mean_abs_error, exact_rate):
    print(f"{amplitude:>17.1f} | {err:>27.3f} | {rate_ok:>24.3f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(amplitudes, mean_abs_error, marker="o")
axes[0].set_title("Средняя абсолютная ошибка обнаружения шаблона")
axes[0].set_ylabel("Ошибка, отсчёты")

axes[1].plot(amplitudes, exact_rate, marker="s")
axes[1].set_title("Доля точного обнаружения шаблона")
axes[1].set_xlabel("Амплитуда шаблона")
axes[1].set_ylabel("Доля точных оценок")

plt.tight_layout()
plt.show()

example_amplitude = 1.0
rng = np.random.default_rng(42)
long_signal = rng.standard_normal(long_len)
true_pos = int(rng.integers(0, long_len - template_len + 1))
long_signal[true_pos:true_pos + template_len] += example_amplitude * template
corr = np.correlate(long_signal, template, mode="valid")
est_pos = int(np.argmax(corr))

fig, axes = plt.subplots(3, 1, figsize=(14, 9))

axes[0].plot(template)
axes[0].set_title("Шаблон")
axes[0].set_xlabel("Отсчёт")
axes[0].set_ylabel("Амплитуда")

axes[1].plot(long_signal, label="Длинный сигнал")
axes[1].axvline(true_pos, color="g", linestyle=":", label="Истинная позиция")
axes[1].axvline(est_pos, color="r", linestyle="--", label="Найденная позиция")
axes[1].set_title("Исходный длинный сигнал")
axes[1].set_xlabel("Отсчёт")
axes[1].set_ylabel("Амплитуда")
axes[1].legend()

axes[2].plot(corr, label="Кросс-корреляция")
axes[2].axvline(est_pos, color="r", linestyle="--", label="Пик корреляции")
axes[2].set_title("Результат кросс-корреляции")
axes[2].set_xlabel("Позиция начала шаблона")
axes[2].set_ylabel("Корреляция")
axes[2].legend()

plt.tight_layout()
plt.show()


**вывод: с ростом амплитуды шаблона точность обнаружения заметно повышается, и при достаточно большом отношении сигнал/шум пик кросс-корреляции уверенно указывает позицию вставки.**


## 6. Работа с аудио в Python

В этом пункте используется указанный WAV-файл `sample-3s.wav`.


In [ ]:
from IPython.display import Audio, display

!wget -q -O sample-3s.wav https://raw.githubusercontent.com/ItserX/dsp-seminars/main/data/sample-3s.wav

rate, data = wavfile.read("sample-3s.wav")

if data.ndim == 2:
    audio = data.mean(axis=1)
else:
    audio = data

audio = audio.astype(np.float64) / 32767.0

fragment_start = int(0.9 * rate)
fragment_len = int(0.35 * rate)
fragment = audio[fragment_start:fragment_start + fragment_len]

corr_audio = signal.correlate(audio, fragment, mode="valid")
estimated_start = int(np.argmax(corr_audio))
estimated_end = estimated_start + fragment_len

print(f"Частота дискретизации: {rate} Гц")
print(f"Длина сигнала: {len(audio)} отсчётов ({len(audio) / rate:.2f} с)")
print(f"Позиция фрагмента в отсчётах: {estimated_start}")
print(f"Позиция фрагмента в секундах: {estimated_start / rate:.4f}")

fig, axes = plt.subplots(3, 1, figsize=(14, 9))

axes[0].plot(audio, label="Полный аудиосигнал")
axes[0].axvline(fragment_start, color="g", linestyle=":", label="Исходная позиция фрагмента")
axes[0].axvline(estimated_start, color="r", linestyle="--", label="Найденная позиция")
axes[0].set_title("Полный аудиосигнал")
axes[0].set_xlabel("Отсчёт")
axes[0].set_ylabel("Амплитуда")
axes[0].legend()

axes[1].plot(fragment, label="Фрагмент")
axes[1].set_title("Фрагмент аудиосигнала")
axes[1].set_xlabel("Отсчёт")
axes[1].set_ylabel("Амплитуда")
axes[1].legend()

axes[2].plot(corr_audio, label="Кросс-корреляция")
axes[2].axvline(estimated_start, color="r", linestyle="--", label="Пик корреляции")
axes[2].set_title("Кросс-корреляция полного сигнала и фрагмента")
axes[2].set_xlabel("Позиция начала фрагмента")
axes[2].set_ylabel("Корреляция")
axes[2].legend()

plt.tight_layout()
plt.show()

display(Audio(audio, rate=rate))
display(Audio(fragment, rate=rate))
display(Audio(audio[estimated_start:estimated_end], rate=rate))


**вывод: максимум кросс-корреляции точно находит выбранный фрагмент в аудиосигнале, а вырезанный по найденной позиции участок совпадает с исходным фрагментом.**


## 7. Рекомендации по выполнению заданий

- Для воспроизводимости результатов используйте `np.random.seed(42)` при генерации случайных чисел.
- Все графики должны быть подписаны (оси, заголовки, легенды).
- При анализе влияния шума на точность выполняйте несколько прогонов с разным `np.random.seed()` и усредняйте результаты.
- Для визуализации корреляции полезно показывать и сам сигнал, и корреляционную функцию.


## 8. Типичные ошибки и их избегание

1. **Путаница режимов `np.convolve`:** всегда проверяйте длину результата.
2. **Краевые эффекты:** при `mode='full'` корреляция даёт значения для всех возможных сдвигов, включая те, где перекрытие сигналов мало. Для поиска задержки это нормально, но при интерпретации нужно учитывать.
3. **Неправильное вычисление сдвига:** используйте `delay = argmax(corr) - (len(x)-1)`.
4. **Ненормализованное аудио:** перед прослушиванием убедитесь, что значения находятся в диапазоне [-1, 1].
5. **Сравнение с аналитикой:** при свёртке гауссовых функций не забывайте нормировать окна (сумма коэффициентов = 1), чтобы амплитуды соответствовали теоретическим.
